# Torchvision detection through Klygo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/integration/Torchvision_COCO15.ipynb)


Run this notebook from the current Klygo checkout after building the real COCO15 fixture. It uses four COCO classes and writes generated results under an ignored artifact root. Torchvision supplies the detection model; Klygo handles data, preprocessing, outputs, postprocessing, visualization, and files.


In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('klygo') is None:
    raise RuntimeError('Run from the current Klygo source checkout')
if importlib.util.find_spec('torchvision') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.[torchvision]'])


In [ ]:
from torchvision.models import get_model, get_model_weights
from klygo import archive, config, datasets, files, media, outputs, postprocessing, preprocessing, visual
from examples.integration.torchvision_coco15 import evaluate, CLASSES, TorchvisionCOCODetect
dataset_root = files.path('artifacts/integration/coco15')
artifact_root = files.path('artifacts/integration/torchvision-coco15-notebook')
files.mkdir(artifact_root, parents=True, exist_ok=True)
assert files.is_file(files.join(dataset_root, 'data.yaml')), 'Build COCO15 first'
print('Classes:', CLASSES)


## Jump to an API

- [`Detector`](#detector): connect an application model through the Torchvision backend.
- [`models.Detector.predict`](#modelsdetectorpredict): run real inference in the combined workflow.


### Detector

The application-owned `TorchvisionCOCODetect(Detector)` class chooses how to construct the model and map class IDs. Klygo only supplies the shared Torchvision backend hooks. This untrained instance checks the wiring without a weight download; the real workflow below uses pretrained weights.


In [ ]:
architecture = 'ssdlite320_mobilenet_v3_large'
module = get_model(architecture, weights=None, weights_backbone=None)
categories = get_model_weights(architecture).DEFAULT.meta['categories']
model = TorchvisionCOCODetect(module, categories)
print(model.backend, len(model.categories), model.current_device())
assert 'person' in model.categories.values()
model.unload()


### models.Detector.predict

`predict(source, prompt=None, **kwargs)` returns Klygo `Detections`. Here `prompt` filters the fixed COCO vocabulary; an unknown class raises `ValueError`. The combined workflow below uses pretrained weights and one positive plus one negative image per class. It reports pixel-space XYXY matches at confidence 0.25 and IoU 0.5.

## Put it together


In [ ]:
metadata = config.load(files.join(dataset_root, 'data.yaml'), verbose=False)
truth = datasets.load(dataset_root, format='yolo', verbose=False)
print('Dataset images:', len(truth), 'named classes:', len(metadata.names))
report = evaluate(dataset_root=dataset_root, artifact_root=artifact_root)
assert report['images'] == 8
print('TP/FP/FN:', report['totals'])


In [ ]:
saved = files.load(files.join(artifact_root, 'report', 'metrics.json'), verbose=False)
print('Backend:', saved['backend'], 'device:', saved['device'])
print('Source digest:', saved['source_sha256'])
print('Archive members:', archive.list_files(files.join(artifact_root, 'report.zip')))
preview = media.load(files.join(artifact_root, 'preview', 'person.jpg'), verbose=False)[0]
display(preview.to_pil())


## Remember

Torchvision supplies fixed COCO labels, not zero-shot classes. Klygo owns the surrounding file, dataset, media, preprocessing, prediction-output, postprocessing, and review path. The 8 image-class runs are a smoke check, not a model benchmark. See [Models Overview](../models/Document_Models_Overview.ipynb), [Models Load](../models/Document_Models_Load.ipynb), and the [workflow guide](../../docs/integration/torchvision-coco15.md).
